# ECLAT (Equivalence Class Clustering and bottom-up Lattice Traversal) - Starter Notebook
ECLAT uses a vertical data format (TID-lists) to find frequent itemsets efficiently.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
# Horizontal format
transactions = [
    [1, 'A','B','C'],
    [2, 'A','C'],
    [3, 'A','B'],
    [4, 'B','C','D'],
    [5, 'A','B','C','D'],
    [6, 'B','D'],
    [7, 'A','C','D'],
    [8, 'A','B','D'],
    [9, 'C','D'],
    [10,'A','B','C','D'],
]

N = len(transactions)

# Build vertical TID-lists
tid_lists = defaultdict(set)
for row in transactions:
    tid = row[0]
    for item in row[1:]:
        tid_lists[frozenset([item])].add(tid)

print('Vertical TID-lists:')
for k, v in sorted(tid_lists.items(), key=lambda x: str(sorted(x[0]))):
    print(f'  {set(k)}: {sorted(v)}')

In [ ]:
MIN_SUPPORT = 0.4
min_count = int(MIN_SUPPORT * N)

def eclat(prefix, items, min_count, frequent):
    while items:
        item, tids = items.pop(0)
        new_itemset = prefix | item
        if len(tids) >= min_count:
            frequent[new_itemset] = len(tids) / N
            # Intersect with remaining items
            suffix = []
            for other_item, other_tids in items:
                new_tids = tids & other_tids
                if len(new_tids) >= min_count:
                    suffix.append((other_item, new_tids))
            eclat(new_itemset, suffix, min_count, frequent)

# Filter 1-itemsets
items_list = [(k, v) for k, v in tid_lists.items() if len(v) >= min_count]
items_list.sort(key=lambda x: str(sorted(x[0])))

frequent = {}
eclat(frozenset(), list(items_list), min_count, frequent)

print(f'\nFrequent itemsets (min_support={MIN_SUPPORT}):')
for fs, sup in sorted(frequent.items(), key=lambda x: (len(x[0]), -x[1])):
    print(f'  {set(fs):30s}  support={sup:.2f}')

In [ ]:
# Summary: count by itemset size
size_counts = defaultdict(int)
for fs in frequent:
    size_counts[len(fs)] += 1

sizes = sorted(size_counts.keys())
counts = [size_counts[s] for s in sizes]

plt.figure(figsize=(6, 4))
plt.bar([str(s) for s in sizes], counts, color='teal', edgecolor='k')
plt.xlabel('Itemset size')
plt.ylabel('Count of frequent itemsets')
plt.title(f'ECLAT: Frequent Itemsets by Size (min_support={MIN_SUPPORT})')
plt.tight_layout()
plt.show()

In [ ]:
# Compare ECLAT vs Apriori approach on support values
print('Top-5 itemsets by support:')
top5 = sorted(frequent.items(), key=lambda x: -x[1])[:5]
for fs, sup in top5:
    print(f'  {set(fs)} -> support={sup:.2f}')